Взрываемся

In [ ]:
import pandas as pd

melb_df = pd.read_csv('data/melb_data_fe.csv')
# melb_df.head()

In [ ]:
display(melb_df.info())

In [ ]:
# Преобразование столбца Date в формат datetime
melb_df['Date'] = pd.to_datetime(melb_df['Date'])

# Выделение квартала продажи объектов недвижимости
melb_df['Quarter'] = melb_df['Date'].dt.quarter

# Преобразование столбца Date в формат datetime
melb_df['Date'] = pd.to_datetime(melb_df['Date'])

# Преобразование столбцов с менее чем 150 уникальными значениями в тип category
excluded_columns = ['Date', 'Rooms', 'Bedroom', 'Bathroom', 'Car']
for col in melb_df.columns:
    if melb_df[col].nunique() < 150 and col not in excluded_columns:
        melb_df[col] = melb_df[col].astype('category')


# Фильтрация таунхаусов с количеством комнат больше 2
filtered_df = melb_df[(melb_df['Type'] == 'townhouse') & (melb_df['Rooms'] > 2)]

# Сортировка по возрастанию числа комнат и по убыванию средней площади комнат
sorted_df = filtered_df.sort_values(by=['Rooms', 'MeanRoomsSquare'], ascending=[True, False])

# Сброс индексов
sorted_df.reset_index(drop=True, inplace=True)

# Цена объекта в строке 18
price = sorted_df.loc[18, 'Price']

print(int(price))



In [ ]:
# # Подсчет количества столбцов типа category
# category_columns_count = melb_df.select_dtypes(include='category').shape[1] - 1

# # Вывод результата
# category_columns_count

In [ ]:
# # Сортировка по столбцу AreaRatio по убыванию
# melb_df = melb_df.sort_values(by='AreaRatio', ascending=False)

# # Сортировка по площади здания (Area) в порядке убывания
# melb_df = melb_df.sort_values(by='BuildingArea', ascending=False)

# # Замена индексов таблицы на новые
# melb_df.reset_index(drop=True, inplace=True)

# # Получение значения площади здания в строке 1558 и округление до целого числа
# area_value = round(melb_df.loc[1558, 'BuildingArea'])
# print(area_value)


In [ ]:
# melb_df.groupby(by='Type').mean(numeric_only=True)

# melb_df.groupby('Type')['Price'].mean()

# melb_df.groupby('Regionname')['Distance'].min().sort_values(ascending=False)

# melb_df.groupby('MonthSale')['Price'].agg(
#     ['count', 'mean', 'max']
# ).sort_values(by='count', ascending=False)

# melb_df.groupby('MonthSale')['Price'].agg(
#     ['count', 'mean', 'max']
# ).sort_values(by='count', ascending=False)

melb_df.groupby('MonthSale')['Price'].agg('describe')

In [ ]:
"""
Ячейка, созданная Data Wrangler.
"""
def clean_data(melb_df):
    melb_df.groupby(by='Type').mean(numeric_only=True)
    # Группировка по столбцам: 'Type'
    melb_df = melb_df.groupby(['Type']).count().reset_index()[['Type']]
    return melb_df

melb_df_clean = clean_data(melb_df.copy())
melb_df_clean.head()

In [ ]:
# Группировка данных по количеству комнат и расчет средней цены в каждой группе
grouped_df = melb_df.groupby('Rooms')['Price'].mean()

# Нахождение количества комнат с наибольшей средней ценой
max_avg_price_rooms = grouped_df.idxmax()

print(max_avg_price_rooms)

In [ ]:
# Группировка данных по регионам и расчет стандартного отклонения широты
grouped_df = melb_df.groupby('Regionname')['Lattitude'].std()

# Нахождение региона с наименьшим стандартным отклонением
min_std_region = grouped_df.idxmin()

print(min_std_region)

In [ ]:
# Фильтрация данных по датам с 1 мая по 1 сентября 2017 года (включительно)
filtered_df = melb_df[(melb_df['Date'] >= '2017-05-01') & (melb_df['Date'] <= '2017-09-01')]

# Группировка данных по компаниям и расчет суммы продаж
grouped_df = filtered_df.groupby('SellerG')['Price'].sum()

# Нахождение компании с наименьшей суммой продаж
min_revenue_company = grouped_df.idxmin()

print(min_revenue_company)

In [ ]:
melb_df.groupby(['Rooms', 'Type'])['Price'].mean().unstack()

In [ ]:
melb_df.pivot_table(
    values='Landsize',
    index='Regionname',
    columns='Type',
    aggfunc=['median', 'mean'],
    fill_value=0
)

In [ ]:
melb_df.pivot_table(
    values='Price',
    index=['Method','Type'],
    columns='Regionname',
    aggfunc='median',
    fill_value=0
)

In [ ]:
# Создаем сводную таблицу
pivot_table = melb_df.pivot_table(
    values='BuildingArea',
    index='Rooms',
    columns='Type',
    aggfunc='median'
)

# # Округляем значения (если требуется)
# pivot_table_rounded = pivot_table.round()

# # Находим комбинацию признаков с наибольшей медианной площадью здания
# max_value = pivot_table_rounded.max().max()
# max_combination = pivot_table_rounded.stack().idxmax()

# print("Комбинация с наибольшей медианной площадью здания:", max_combination)

display(pivot_table)


In [ ]:
# Создаем сводную таблицу
pivot_table = melb_df.pivot_table(
    values='Price',
    index='SellerG',
    columns='Type',
    aggfunc='median'
)

# Находим агентство с максимальной медианной ценой для зданий типа 'unit'
max_median_price_unit = pivot_table['unit'].max()
agency_with_max_price_unit = pivot_table['unit'].idxmax()

display(pivot_table)
print("Агентство с максимальной медианной ценой для зданий типа 'unit':", agency_with_max_price_unit)
